# Training Scaffold

In [1]:
import os
import json
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# -----------------------
# 1. Dataset
# -----------------------
class WheelHeatmapDataset(Dataset):
    def __init__(self, json_path, img_dir, img_size=128, sigma=3):
        with open(json_path) as f:
            self.labels = json.load(f)
        self.img_dir = Path(img_dir)
        self.img_size = img_size
        self.sigma = sigma
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        entry = self.labels[idx]
        img_path = self.img_dir / entry["filename"]
        img = Image.open(img_path).convert("RGB")
        img_t = self.transform(img)

        # Create target heatmap
        heatmap = np.zeros((self.img_size, self.img_size), dtype=np.float32)
        if entry["is_present"] == 1.0:
            cx = int(entry["center"][0] * self.img_size)
            cy = int(entry["center"][1] * self.img_size)
            heatmap = self._draw_gaussian(heatmap, (cx, cy), self.sigma)

        return img_t, torch.tensor(heatmap).unsqueeze(0), entry["filename"]

    def _draw_gaussian(self, heatmap, center, sigma):
        """Draw a 2D Gaussian on the heatmap."""
        x, y = center
        h, w = heatmap.shape
        xs = np.arange(0, w, 1, np.float32)
        ys = np.arange(0, h, 1, np.float32)[:, np.newaxis]
        g = np.exp(-((xs - x)**2 + (ys - y)**2) / (2 * sigma**2))
        heatmap = np.maximum(heatmap, g)
        return heatmap

# -----------------------
# 2. Model (U-Net style, full 128×128 output)
# -----------------------
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: ResNet18 backbone
        self.encoder = models.resnet18(pretrained=True)
        self.encoder.fc = nn.Identity()
        self.enc_layers = list(self.encoder.children())[:-2]
        self.encoder = nn.Sequential(*self.enc_layers)

        # Decoder: upsample back to 128×128
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),  # 4→8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 2, stride=2),  # 8→16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 2, stride=2),   # 16→32
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 2, stride=2),    # 32→64
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, 2, stride=2),    # 64→128
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=1)             # output heatmap
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x  # (B,1,128,128)

# -----------------------
# 3. Training Loop
# -----------------------
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for imgs, heatmaps, _ in train_loader:  # ignore filenames
            imgs, heatmaps = imgs.to(device), heatmaps.to(device)
            preds = model(imgs)
            loss = criterion(preds, heatmaps)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        val_loss = 0
        model.eval()
        with torch.no_grad():
            for imgs, heatmaps, _ in val_loader:  # ignore filenames
                imgs, heatmaps = imgs.to(device), heatmaps.to(device)
                preds = model(imgs)
                loss = criterion(preds, heatmaps)
                val_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f}")

# -----------------------
# 4. Usage
# -----------------------
if __name__ == "__main__":
    base_dir = Path.cwd()
    img_dir = base_dir / "labeled_example_data"
    labels_path = img_dir / "labels.json"

    dataset = WheelHeatmapDataset(labels_path, img_dir, img_size=128, sigma=3)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SimpleUNet()
    train_model(model, train_loader, val_loader, epochs=20, lr=1e-3, device=device)

/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20 | Train Loss: 0.0025 | Val Loss: 0.0015
Epoch 2/20 | Train Loss: 0.0015 | Val Loss: 0.0014
Epoch 3/20 | Train Loss: 0.0014 | Val Loss: 0.0013
Epoch 4/20 | Train Loss: 0.0013 | Val Loss: 0.0013
Epoch 5/20 | Train Loss: 0.0012 | Val Loss: 0.0013
Epoch 6/20 | Train Loss: 0.0010 | Val Loss: 0.0009
Epoch 7/20 | Train Loss: 0.0008 | Val Loss: 0.0007
Epoch 8/20 | Train Loss: 0.0006 | Val Loss: 0.0007
Epoch 9/20 | Train Loss: 0.0005 | Val Loss: 0.0006
Epoch 10/20 | Train Loss: 0.0004 | Val Loss: 0.0006
Epoch 11/20 | Train Loss: 0.0004 | Val Loss: 0.0005
Epoch 12/20 | Train Loss: 0.0003 | Val Loss: 0.0005
Epoch 13/20 | Train Loss: 0.0003 | Val Loss: 0.0004
Epoch 14/20 | Train Loss: 0.0003 | Val Loss: 0.0005
Epoch 15/20 | Train Loss: 0.0003 | Val Loss: 0.0004
Epoch 16/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 17/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 18/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 19/20 | Train Loss: 0.0002 | Val Loss: 0.0004
Epoch 20/20 | Train L

In [4]:
import os
import csv
import cv2
import numpy as np
import torch
from pathlib import Path

# Config
save_dir = Path.cwd() / "eval_outputs"
overlay_dir = save_dir / "overlays"
heatmap_dir = save_dir / "heatmaps"
worst_dir = save_dir / "worst_overlays"
save_dir.mkdir(exist_ok=True)
overlay_dir.mkdir(parents=True, exist_ok=True)
heatmap_dir.mkdir(parents=True, exist_ok=True)
worst_dir.mkdir(parents=True, exist_ok=True)

# If not set earlier, define these:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# img_dir = Path.cwd() / "labeled_example_data"

def to_uint8(img):
    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

def colorize_heatmap(hm, cmap=cv2.COLORMAP_JET, normalize=True):
    # hm expected float32, shape (H, W)
    hm = hm.copy()
    if normalize:
        mmin, mmax = hm.min(), hm.max()
        if mmax > mmin:
            hm = (hm - mmin) / (mmax - mmin)
        else:
            hm = np.zeros_like(hm)
    hm_color = cv2.applyColorMap((hm * 255).astype(np.uint8), cmap)
    return hm_color

def overlay_heatmap_on_image(img_bgr, hm_color, alpha=0.4):
    # img_bgr and hm_color must be same size
    return cv2.addWeighted(img_bgr, 1.0, hm_color, alpha, 0)

# Evaluate
model.eval()
all_results = []

with torch.no_grad():
    for batch in val_loader:
        # val_loader returns (imgs, heatmaps, filenames)
        if len(batch) == 3:
            imgs, heatmaps, fnames = batch
        else:
            # Fallback if filenames not provided
            imgs, heatmaps = batch
            fnames = [f"val_{i:06d}.png" for i in range(imgs.size(0))]

        imgs = imgs.to(device)
        heatmaps = heatmaps.to(device)

        preds = model(imgs)  # (B,1,128,128)
        preds_np = preds.squeeze(1).cpu().numpy()   # (B,128,128)
        gts_np   = heatmaps.squeeze(1).cpu().numpy()  # (B,128,128)

        B = imgs.size(0)
        for b in range(B):
            pred_hm = preds_np[b]
            gt_hm   = gts_np[b]
            fname   = fnames[b]

            # Predicted center (pixels in 128x128 space)
            py, px = np.unravel_index(np.argmax(pred_hm), pred_hm.shape)
            # Ground-truth center
            gy, gx = np.unravel_index(np.argmax(gt_hm), gt_hm.shape)

            err = float(np.hypot(px - gx, py - gy))

            all_results.append({
                "filename": fname,
                "gt_x": int(gx), "gt_y": int(gy),
                "pred_x": int(px), "pred_y": int(py),
                "error_px": err
            })

            # Load original image, resize to 128x128 (to match coordinates)
            img_path = img_dir / fname
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                # If missing, create a blank canvas for consistency
                img_bgr = np.zeros((128, 128, 3), dtype=np.uint8)
            else:
                img_bgr = cv2.resize(img_bgr, (128, 128), interpolation=cv2.INTER_LINEAR)

            # Draw GT (red) and prediction (green)
            vis = img_bgr.copy()
            cv2.circle(vis, (gx, gy), 5, (0, 0, 255), -1)   # GT red filled
            cv2.circle(vis, (px, py), 5, (0, 255, 0), 2)    # Pred green ring
            cv2.putText(vis, f"err={err:.2f}px", (8, 22),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

            # Save overlay
            cv2.imwrite(str(overlay_dir / f"overlay_{Path(fname).name}"), vis)

            # Optional: heatmap overlay for prediction
            pred_hm_color = colorize_heatmap(pred_hm)
            hm_overlay = overlay_heatmap_on_image(img_bgr, pred_hm_color, alpha=0.45)
            cv2.circle(hm_overlay, (gx, gy), 4, (0, 0, 255), -1)
            cv2.circle(hm_overlay, (px, py), 4, (0, 255, 0), 2)
            cv2.imwrite(str(heatmap_dir / f"hm_{Path(fname).name}"), hm_overlay)

# Metrics
errors = np.array([r["error_px"] for r in all_results], dtype=np.float32)
mean_err = float(np.mean(errors)) if len(errors) else float("nan")
median_err = float(np.median(errors)) if len(errors) else float("nan")
pct_2 = float(np.mean(errors <= 2) * 100) if len(errors) else float("nan")
pct_5 = float(np.mean(errors <= 5) * 100) if len(errors) else float("nan")

print(f"✅ Mean pixel error: {mean_err:.2f}")
print(f"📏 Median pixel error: {median_err:.2f}")
print(f"🎯 % within 2px: {pct_2:.1f}%")
print(f"🎯 % within 5px: {pct_5:.1f}%")
print(f"🖼 Overlays: {overlay_dir}")
print(f"🔥 Heatmap overlays: {heatmap_dir}")

# Save CSV
csv_path = save_dir / "eval_results.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["filename", "gt_x", "gt_y", "pred_x", "pred_y", "error_px"])
    for r in all_results:
        writer.writerow([r["filename"], r["gt_x"], r["gt_y"], r["pred_x"], r["pred_y"], f"{r['error_px']:.4f}"])
print(f"📄 CSV saved: {csv_path}")

# Save top-N worst cases for quick review
N = min(30, len(all_results))
worst_idx = np.argsort(errors)[-N:][::-1] if len(errors) else []
for i in worst_idx:
    fname = all_results[i]["filename"]
    src = overlay_dir / f"overlay_{Path(fname).name}"
    dst = worst_dir / f"{errors[i]:.2f}px__{Path(fname).name}"
    if src.exists():
        img = cv2.imread(str(src))
        cv2.imwrite(str(dst), img)
print(f"⚠️ Worst-{N} overlays saved to: {worst_dir}")

✅ Mean pixel error: 2.02
📏 Median pixel error: 1.41
🎯 % within 2px: 72.5%
🎯 % within 5px: 95.0%
🖼 Overlays: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/eval_outputs/overlays
🔥 Heatmap overlays: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/eval_outputs/heatmaps
📄 CSV saved: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/eval_outputs/eval_results.csv
⚠️ Worst-30 overlays saved to: /Users/amrattia/projects/ATI Perception Take Home Challenge/Challenge 1 - Rim Center Location/eval_outputs/worst_overlays
